# AuraShield: Deepfake Audio Detection Pipeline

This notebook contains the complete, reproducible pipeline for training and evaluating models to distinguish between **Genuine Human Speech** and **AI-Generated Deepfake Audio**.

### Pipeline Stages:
1. **Exploratory Data Analysis (EDA)**: Inspecting audio properties.
2. **Feature Extraction**: Tabular aggregates of MFCCs/spectral parameters and 2D Log-Mel Spectrograms.
3. **Model Training**: LightGBM Classifier and PyTorch 2D CNN with feature caching.
4. **Evaluation**: EER (Equal Error Rate), Accuracy, F1-Score, and recall performance.

## 1. Imports and Reproducibility Setup

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import torch

from src.features import load_and_normalize_audio, extract_tabular_features, extract_mel_spectrogram
from src.models import AudioCNN, InMemoryDataset, compute_eer
from train_pipeline import set_seed, load_dataset_parallel

# Set seed for reproducibility
set_seed(42)
print("PyTorch version:", torch.__version__)
print("Device count:", 1 if torch.cuda.is_available() else 0, "(Using CPU for training)")

## 2. Exploratory Data Analysis & Visualizations

Let's load a sample audio file from both the `real` and `fake` classes to visualize their time-domain waveforms and Mel Spectrograms.

In [ ]:
sample_real = os.path.join("dataset", "real", os.listdir(os.path.join("dataset", "real"))[0])
sample_fake = os.path.join("dataset", "fake", os.listdir(os.path.join("dataset", "fake"))[0])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Real Audio Waveform & Mel Spectrogram
y_real, sr = load_and_normalize_audio(sample_real)
librosa.display.waveshow(y_real, sr=sr, ax=axes[0, 0], color='#10b981')
axes[0, 0].set_title("Genuine Audio Waveform")

S_real = extract_mel_spectrogram(y_real, sr)
librosa.display.specshow(S_real, sr=sr, x_axis='time', y_axis='mel', ax=axes[1, 0], cmap='viridis')
axes[1, 0].set_title("Genuine Mel Spectrogram")

# Fake Audio Waveform & Mel Spectrogram
y_fake, sr = load_and_normalize_audio(sample_fake)
librosa.display.waveshow(y_fake, sr=sr, ax=axes[0, 1], color='#ef4444')
axes[0, 1].set_title("Deepfake Audio Waveform")

S_fake = extract_mel_spectrogram(y_fake, sr)
librosa.display.specshow(S_fake, sr=sr, x_axis='time', y_axis='mel', ax=axes[1, 1], cmap='magma')
axes[1, 1].set_title("Deepfake Mel Spectrogram")

plt.tight_layout()
plt.show()

## 3. Load Cached Dataset Features

To save time, we load our pre-extracted features cache from `models/feature_cache.pkl` which stores processed features for all 13,956 files.

In [ ]:
X_tab, X_mel, y, file_paths = load_dataset_parallel("dataset")
print(f"Loaded {len(y)} samples.")
print("Tabular features shape:", X_tab.shape)
print("Mel Spectrogram shape:", X_mel.shape)

## 4. Train-Test Splits

In [ ]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(y))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])

X_train_tab, y_train = X_tab[idx_train], y[idx_train]
X_val_tab, y_val = X_tab[idx_val], y[idx_val]
X_test_tab, y_test = X_tab[idx_test], y[idx_test]

X_train_mel = X_mel[idx_train]
X_val_mel = X_mel[idx_val]
X_test_mel = X_mel[idx_test]

## 5. LightGBM Model Evaluation

We load the trained LightGBM model to perform predictions on our test split.

In [ ]:
lgb_save_path = os.path.join('models', 'lightgbm_model.pkl')
with open(lgb_save_path, 'rb') as f:
    lgb_model = pickle.load(f)
    
lgb_probs = lgb_model.predict_proba(X_test_tab)[:, 1]
lgb_preds = (lgb_probs >= 0.5).astype(int)
lgb_acc = np.mean(lgb_preds == y_test)
lgb_eer = compute_eer(y_test, lgb_probs)

print(f"LightGBM Test Accuracy: {lgb_acc*100:.2f}%")
print(f"LightGBM Test EER:      {lgb_eer*100:.2f}%")

## 6. PyTorch CNN Model Evaluation

Load the trained CNN weights and score the Test split.

In [ ]:
from torch.utils.data import DataLoader
test_dataset = InMemoryDataset(X_test_mel, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn_model = AudioCNN()
cnn_model.load_state_dict(torch.load(os.path.join('models', 'cnn_model.pth'), map_location=device))
cnn_model.to(device)
cnn_model.eval()

cnn_probs = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        logits = cnn_model(X_batch)
        probs = torch.sigmoid(logits).cpu().numpy().squeeze()
        cnn_probs.extend(probs)

cnn_probs = np.array(cnn_probs)
cnn_preds = (cnn_probs >= 0.5).astype(int)
cnn_acc = np.mean(cnn_preds == y_test)
cnn_eer = compute_eer(y_test, cnn_probs)

print(f"PyTorch CNN Test Accuracy: {cnn_acc*100:.2f}%")
print(f"PyTorch CNN Test EER:      {cnn_eer*100:.2f}%")

## 7. Ensemble Evaluation

In [ ]:
ensemble_probs = (lgb_probs + cnn_probs) / 2.0
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

ens_acc = np.mean(ensemble_preds == y_test)
ens_eer = compute_eer(y_test, ensemble_probs)

print(f"Ensemble Test Accuracy: {ens_acc*100:.2f}%")
print(f"Ensemble Test EER:      {ens_eer*100:.2f}%")

## 8. ROC curves and Confusion Matrix Plotting

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Plot Confusion Matrix
cm = confusion_matrix(y_test, ensemble_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Genuine', 'Deepfake'], yticklabels=['Genuine', 'Deepfake'])
plt.title('Ensemble Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Plot ROC Curves
plt.figure(figsize=(7, 5))
for probs, name in zip([lgb_probs, cnn_probs, ensemble_probs], ['LightGBM', 'CNN', 'Ensemble']):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curves comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()